## Import of Libraries

In [1]:
# Import visualization libraries
import seaborn as sns
import matplotlib.pyplot as plt

# Import data handling libraries
import pandas as pd
import numpy as np
import pickle

# Import machine learning libraries
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

## Configuration of Display Options

In [2]:
# Set display format for floats to 2 decimal places
pd.set_option("display.float_format", "{:,.2f}".format)

## Loading of Dataset

In [3]:
# Load dataset
df = pd.read_csv("../data/raw/-spotify-tracks-dataset/dataset.csv")

df = df.copy()

## Cleanup and Datatype Transformation

In [4]:
# Drop unwanted column Unnamed: 0
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

# Transform datatype of column explicit to int
df["explicit"] = df["explicit"].astype(int)

# Transform datatype of column duration_ms to minutes
df["duration_min"] = df["duration_ms"] / 60000

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   track_id          114000 non-null  object 
 1   artists           113999 non-null  object 
 2   album_name        113999 non-null  object 
 3   track_name        113999 non-null  object 
 4   popularity        114000 non-null  int64  
 5   duration_ms       114000 non-null  int64  
 6   explicit          114000 non-null  int64  
 7   danceability      114000 non-null  float64
 8   energy            114000 non-null  float64
 9   key               114000 non-null  int64  
 10  loudness          114000 non-null  float64
 11  mode              114000 non-null  int64  
 12  speechiness       114000 non-null  float64
 13  acousticness      114000 non-null  float64
 14  instrumentalness  114000 non-null  float64
 15  liveness          114000 non-null  float64
 16  valence           11

## Handling of Duplicates, Missing Values, 0 Values and Outliers

In [5]:
# No duplicates contained

# Imputation of missing values
missing_values = df[df.isnull().any(axis=1)]
display(missing_values)

columns_to_fill = ['artists', 'album_name', 'track_name']

for col in columns_to_fill:
    df[col] = df[col].fillna('missing')

display("Missing Values")
df.isna().sum().sum()

# 0 values in numerical features are kept

# Outliers are kept as they represent valid extreme values

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,...,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,duration_min
65900,1kR4gIb7nGxHPI3D2ifs59,NaN,NaN,NaN,0,0,0,0.50,0.58,7,...,0,0.06,0.69,0.00,0.07,0.73,138.39,4,k-pop,0.00


'Missing Values'

np.int64(0)

## Feature Engineering

In [6]:
# Create camelot-ID for Camelot Wheel mapping (mixed in key)
# Key mapping: Create the numbers 1–12 for the 12 keys.
camelot_number_map = {
    0: 8,   # C
    1: 3,   # C#/Db
    2: 10,  # D
    3: 5,   # D#/Eb
    4: 12,  # E
    5: 7,   # F
    6: 2,   # F#/Gb
    7: 9,   # G
    8: 4,   # G#/Ab
    9: 11,  # A
    10: 6,  # A#/Bb
    11: 1   # B
}

# Mode mapping: 1=major (Dur) -> A, 0=minor (Moll) -> B
mode_map = {1: 'A', 0: 'B'}

# Calculate Camelot ID from key and mode
df['camelot_id'] = df['key'].map(camelot_number_map).astype(str) + df['mode'].map(mode_map)

# Check result
display("Camelot-ID mapping")
df[['key', 'mode', 'camelot_id']]

'Camelot-ID mapping'

,key,mode,camelot_id
0,1,0,3B
1,1,1,3A
2,0,1,8A
3,0,1,8A
4,2,1,10A
...,...,...,...
113995,5,1,7A
113996,0,0,8B
113997,0,0,8B
113998,7,1,9A


In [7]:
# Create new features for features with high and medium correlations
df["power_score"] = 0.5*df["energy"] + 0.3*df["loudness"] + 0.2*(1 - df["acousticness"])
df["vocal_score"] = df["speechiness"] * (1 - df["instrumentalness"])
df["party_score"] = df["danceability"] * df["valence"]
df["chill_score"] = df["instrumentalness"] * (1 - df["loudness"]) * (1 - df["valence"])

# Create further new features
df["club_factor"] = df["danceability"] * df["energy"]
df["groove"] = df["tempo"] * df["time_signature"]

In [8]:
# Bin existing features
df["popularity_label"] = pd.qcut(
    df["popularity"], 5,
    labels=["Unknown", "Underground", "Aspiring", "Popular", "Hit"],
    duplicates="drop"
).cat.as_ordered()

df["tempo_label"] = pd.cut(
    df["tempo"],
    bins=[-np.inf, 90, 110, 124, 128, 140, np.inf],
    labels=[
        "<90 Slow",
        "90–110 Downtempo",
        "110–124 Midtempo",
        "124–128 House",
        "128–140 Techno/Trance",
        "≥140 Fast"
    ],
    include_lowest=True
).cat.as_ordered()

In [9]:
# Display min and max values of numerical columns for scaling
display("Min and Max Values of Numerical Columns")
df.describe().T.loc[:, ["min", "max"]]

'Min and Max Values of Numerical Columns'

,min,max
popularity,0.00,100.00
duration_ms,0.00,"5,237,295.00"
explicit,0.00,1.00
danceability,0.00,0.98
energy,0.00,1.00
key,0.00,11.00
loudness,-49.53,4.53
mode,0.00,1.00
speechiness,0.00,0.96
acousticness,0.00,1.00


## Filtering Data

In [10]:
# Unique genres in track_genre
unique_genres = df['track_genre'].unique()

display("Unique Genres")
print(unique_genres)

'Unique Genres'

['acoustic' 'afrobeat' 'alt-rock' 'alternative' 'ambient' 'anime'
 'black-metal' 'bluegrass' 'blues' 'brazil' 'breakbeat' 'british'
 'cantopop' 'chicago-house' 'children' 'chill' 'classical' 'club' 'comedy'
 'country' 'dance' 'dancehall' 'death-metal' 'deep-house' 'detroit-techno'
 'disco' 'disney' 'drum-and-bass' 'dub' 'dubstep' 'edm' 'electro'
 'electronic' 'emo' 'folk' 'forro' 'french' 'funk' 'garage' 'german'
 'gospel' 'goth' 'grindcore' 'groove' 'grunge' 'guitar' 'happy'
 'hard-rock' 'hardcore' 'hardstyle' 'heavy-metal' 'hip-hop' 'honky-tonk'
 'house' 'idm' 'indian' 'indie-pop' 'indie' 'industrial' 'iranian'
 'j-dance' 'j-idol' 'j-pop' 'j-rock' 'jazz' 'k-pop' 'kids' 'latin'
 'latino' 'malay' 'mandopop' 'metal' 'metalcore' 'minimal-techno' 'mpb'
 'new-age' 'opera' 'pagode' 'party' 'piano' 'pop-film' 'pop' 'power-pop'
 'progressive-house' 'psych-rock' 'punk-rock' 'punk' 'r-n-b' 'reggae'
 'reggaeton' 'rock-n-roll' 'rock' 'rockabilly' 'romance' 'sad' 'salsa'
 'samba' 'sertanejo' 'show

In [11]:
# Create list for EDM genres
edm_genres = [
    "ambient", "breakbeat", "club", "deep-house", "dance", "drum-and-bass",
    "edm", "electro", "electronic", "hardstyle", "house", "idm",
    "minimal-techno", "progressive-house", "synth-pop", "techno",
    "trance", "dubstep", "chicago-house"
]

display("Count EDM Genres")
len(edm_genres)

'Count EDM Genres'

19

In [12]:
# Filter DataFrame for EDM genres and time_signature == 4
mask = df["track_genre"].isin(edm_genres) & (df["time_signature"] == 4)
df_filtered = df[mask]

display("Count filtered Data Frame")
len(df_filtered)

'Count filtered Data Frame'

17934

## Data Scaling and One-Hot-Encoding

In [13]:
# Define numerical fetaures for scaling (MinMax for bounded features with a fixed range (e.g., 0–100),
# Standard for unbounded features)
minmax_columns   = ["popularity"]
standard_columns = ["duration_min", "loudness", "tempo", "power_score", "chill_score", "groove"]

# Define categorical features for one-hot encoding
ohe_columns = ["track_genre", "camelot_id", "popularity_label", "tempo_label"]

In [14]:
# Build pipelines for numerical and categorical features
numerical_minmax = Pipeline([("imputer", SimpleImputer(strategy="median")), 
                           ("scaler", MinMaxScaler())])
numerical_standard = Pipeline([("imputer", SimpleImputer(strategy="median")), 
                             ("scaler", StandardScaler())])
categorical = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                        ("ohe", OneHotEncoder(handle_unknown="ignore", 
                                              sparse_output=False))])


## Dimensionality Reduction

In [15]:
# Perform principal component analysis (PCA)
# Define features for PCA groups with high and medium correlations
pca_groups = {
    "pca_elac": ["energy", "loudness", "acousticness"],
    "pca_div":  ["danceability", "instrumentalness", "valence"],
    "pca_liv":  ["loudness", "instrumentalness", "valence"],
}

# Build pipelines for PCA groups
pca_pipes = [
    (name, Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=0.90, random_state=42)),
    ]), cols)
    for name, cols in pca_groups.items()
]

## Preprocessing and Transformation

In [16]:
# Create transformers for numerical and categorical features
base_transformers = [
    ("minmax",   numerical_minmax,   minmax_columns),
    ("standard", numerical_standard, standard_columns),
    ("cat",      categorical,        ohe_columns),
]

# Create transformers for PCA groups
pca_transformers = [(name, pipe, cols) for name, pipe, cols in pca_pipes]

# Create additional transformers for features contained in PCA
raw_columns = ["energy", "acousticness", "danceability", "instrumentalness", "valence"]

raw_transformers = [("raw", "passthrough", raw_columns)]

# Create preprocessor
preprocessor = ColumnTransformer(
    transformers=base_transformers + pca_transformers + raw_transformers,
    remainder="passthrough"
)
preprocessor.set_output(transform="pandas") # transform data as df

# Transform columns
df_preprocessed = preprocessor.fit_transform(df_filtered)

# Get fetaure names
feature_names = preprocessor.get_feature_names_out()

## Data Export

In [ ]:
# Save filtered DataFrame
with open("../data/processed/df_filtered.pkl", "wb") as f:
    pickle.dump(df_filtered, f)

# Save preprocessed DataFrame
with open("../data/processed/df_preprocessed.pkl", "wb") as f:
    pickle.dump(df_preprocessed, f)